### **Dim Customers**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.dimcustomers (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT,
    customer_email STRING,
    customer_name STRING,
    customer_name_upper STRING,
    last_updated DATE,
    process_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_customers_source AS
WITH filtered AS (
    SELECT *
    FROM datamodeling.silver.silver_table
    WHERE last_updated > (
        SELECT COALESCE(MAX(last_updated), '1000-01-01')
        FROM datamodeling.gold.dimcustomers
    )
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY last_updated DESC
           ) rn
    FROM filtered
)
SELECT
    customer_id,
    customer_email,
    customer_name,
    customer_name_upper,
    last_updated,
    current_timestamp() AS process_date
FROM ranked
WHERE rn = 1;

In [0]:
%sql
SELECT * FROM  dim_customers_source

In [0]:
%sql
MERGE INTO datamodeling.gold.dimcustomers t
USING dim_customers_source s
ON t.customer_id = s.customer_id

WHEN MATCHED AND s.last_updated > t.last_updated THEN
UPDATE SET
    t.customer_email = s.customer_email,
    t.customer_name = s.customer_name,
    t.customer_name_upper = s.customer_name_upper,
    t.last_updated = s.last_updated,
    t.process_date = s.process_date

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_email,
    customer_name,
    customer_name_upper,
    last_updated,
    process_date
)
VALUES (
    s.customer_id,
    s.customer_email,
    s.customer_name,
    s.customer_name_upper,
    s.last_updated,
    s.process_date
);

In [0]:
%sql
SELECT * FROM  datamodeling.gold.dimcustomers
ORDER BY customer_id;

### **Dim Products**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.dimproducts (
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id INT,
    product_name STRING,
    product_category STRING,
    last_updated DATE,
    process_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_products_source AS

WITH filtered AS (
    SELECT *
    FROM datamodeling.silver.silver_table
    WHERE last_updated >
    (
        SELECT COALESCE(MAX(last_updated), '1000-01-01')
        FROM datamodeling.gold.dimproducts
    )
),

ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY product_id
               ORDER BY last_updated DESC
           ) rn
    FROM filtered
)

SELECT
    product_id,
    product_name,
    product_category,
    last_updated,
    current_timestamp() AS process_date
FROM ranked
WHERE rn = 1;

In [0]:
%sql
SELECT *
FROM dim_products_source

In [0]:
%sql
MERGE INTO datamodeling.gold.dimproducts t
USING dim_products_source s
ON t.product_id = s.product_id

WHEN MATCHED
AND s.last_updated > t.last_updated
THEN UPDATE SET
    product_name = s.product_name,
    product_category = s.product_category,
    last_updated = s.last_updated,
    process_date = s.process_date

WHEN NOT MATCHED
THEN INSERT (
    product_id,
    product_name,
    product_category,
    last_updated,
    process_date
)
VALUES (
    s.product_id,
    s.product_name,
    s.product_category,
    s.last_updated,
    s.process_date
);

In [0]:
%sql
SELECT *
FROM datamodeling.gold.DimProducts
ORDER BY product_id;

### **Dim Payments**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.dimpayments (
    payment_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    payment_type STRING,
    process_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_payments_source AS
SELECT DISTINCT
    payment_type
FROM datamodeling.silver.silver_table
WHERE payment_type IS NOT NULL;

In [0]:
%sql
SELECT *
FROM dim_payments_source

In [0]:
%sql
MERGE INTO datamodeling.gold.dimpayments t
USING dim_payments_source s
ON t.payment_type = s.payment_type

WHEN NOT MATCHED THEN
INSERT (
    payment_type,
    process_date
)
VALUES (
    s.payment_type,
    current_timestamp());

In [0]:
%sql
SELECT *
FROM datamodeling.gold.DimPayments
ORDER BY payment_type;

### **Dim Region**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.dimregion (
    region_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    country STRING,
    process_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_region_source AS
SELECT DISTINCT
    country
FROM datamodeling.silver.silver_table
WHERE country IS NOT NULL;

In [0]:
%sql
SELECT *
FROM dim_region_source

In [0]:
%sql
MERGE INTO datamodeling.gold.dimregion t
USING dim_region_source s
ON t.country = s.country

WHEN NOT MATCHED THEN
INSERT (
    country,
    process_date
)
VALUES (
    s.country,
    current_timestamp());

In [0]:
%sql
SELECT *
FROM datamodeling.gold.DimRegion
ORDER BY country;

### **Dim Date**

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.dim_date
USING DELTA
AS
SELECT
    date_format(date, 'yyyyMMdd') + 0 AS date_key,
    date AS full_date,
    year(date) AS year,
    month(date) AS month,
    day(date) AS day,
    weekofyear(date) AS week_of_year,
    date_format(date, 'EEEE') AS day_name,
    date_format(date, 'MMM') AS month_name,
    CASE WHEN dayofweek(date) IN (1,7) THEN true ELSE false END AS is_weekend,
    quarter(date) AS quarter,
    current_timestamp() AS process_ts
FROM (
    SELECT explode(
        sequence(
            (SELECT MIN(order_date) FROM datamodeling.silver.silver_table),
            (SELECT MAX(order_date) FROM datamodeling.silver.silver_table),
            interval 1 day
        )
    ) AS date
);

### **FACT TABLE**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.fact_sales (

    order_id INT,

    customer_sk BIGINT,
    product_sk BIGINT,
    payment_sk BIGINT,
    region_sk BIGINT,
    date_key INT,

    quantity INT,
    unit_price DECIMAL(10,2),
    sales_amount DECIMAL(18,2),

    last_updated DATE,
    process_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fact_source AS

SELECT

    f.order_id,

    c.customer_sk,
    p.product_sk,
    py.payment_sk,
    r.region_sk,
    d.date_key,

    f.quantity,
    f.unit_price,
    f.quantity * f.unit_price AS sales_amount,

    f.last_updated,
    current_timestamp() AS process_date

FROM datamodeling.silver.silver_table f

LEFT JOIN datamodeling.gold.dimcustomers c
    ON f.customer_id = c.customer_id

LEFT JOIN datamodeling.gold.dimproducts p
    ON f.product_id = p.product_id

LEFT JOIN datamodeling.gold.dimpayments py
    ON f.payment_type = py.payment_type

LEFT JOIN datamodeling.gold.dimregion r
    ON f.country = r.country

LEFT JOIN datamodeling.gold.dim_date d
    ON f.order_date = d.full_date

WHERE f.last_updated >
(
    SELECT COALESCE(MAX(last_updated), '1000-01-01')
    FROM datamodeling.gold.fact_sales
);

In [0]:
%sql
SELECT *
FROM fact_source

In [0]:
%sql
MERGE INTO datamodeling.gold.fact_sales t
USING fact_source s
ON t.order_id = s.order_id

WHEN MATCHED
AND s.last_updated > t.last_updated
THEN UPDATE SET
    customer_sk = s.customer_sk,
    product_sk = s.product_sk,
    payment_sk = s.payment_sk,
    region_sk = s.region_sk,
    date_key = s.date_key,
    quantity = s.quantity,
    unit_price = s.unit_price,
    sales_amount = s.sales_amount,
    last_updated = s.last_updated,
    process_date = s.process_date

WHEN NOT MATCHED
THEN INSERT (
    order_id,
    customer_sk,
    product_sk,
    payment_sk,
    region_sk,
    date_key,
    quantity,
    unit_price,
    sales_amount,
    last_updated,
    process_date
)
VALUES (
    s.order_id,
    s.customer_sk,
    s.product_sk,
    s.payment_sk,
    s.region_sk,
    s.date_key,
    s.quantity,
    s.unit_price,
    s.sales_amount,
    s.last_updated,
    s.process_date
);

In [0]:
%sql
SELECT *
FROM datamodeling.gold.fact_sales
ORDER BY order_id;